# YOLO11n-Seg DPCA P3/P4/P5 Strong — Mr_TU Data, Grouped Near-Duplicate Split

This notebook trains the indicated configuration on `Mr_TU_ShrimpDiseaseSeg_Dat-Copy` version 1. It creates or reuses a derived grouped near-duplicate split before training.

Prepared dataset: `mrtu_yolo26_grouped_near_duplicate_split`. Run cells from top to bottom.


# DPCA — Dual-Polarity Contrast Attention

Kaggle-standalone novel attention experiment. This notebook keeps the established Roboflow download, grouped shrimp-level no-leakage split, data YAML creation, healthy-negative evaluation and reporting workflow.

It trains the same module twice in order: (1) light augmentation exactly matched to the clean baseline, then (2) strong augmentation exactly matched to the SimAM+CA augmentation notebook. The custom module is defined and registered inline, then checked at multiple channels/spatial sizes and by a YOLO build+forward pass before training.

Idea: Models bright and dark local deviations separately, then learns a per-channel soft choice between them.


In [ ]:
# Roboflow download is configured in the following cell via Kaggle Secret ROBOFLOW_API_KEY.


In [ ]:
# /kaggle/working/mrtu_grouped_near_duplicate_split_output


In [ ]:
# Download Mr_TU_ShrimpDiseaseSeg_Dat-Copy v1 as a YOLO26 segmentation export.
# Store the Roboflow key in a Kaggle Secret named ROBOFLOW_API_KEY (or in the
# environment); never paste a real key into this notebook.
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else (
    Path('/content') if Path('/content').exists() else Path.cwd()
)
ROBOFLOW_WORKSPACE = 'lets-try-this'
# Roboflow API requires the project slug, not its display name.
ROBOFLOW_PROJECT = 'mr_tu_shrimpdiseaseseg_dat-copy'
EXPECTED_PROJECT_ID = 'lets-try-this/mr_tu_shrimpdiseaseseg_dat-copy'
EXPECTED_PROJECT_TYPE = 'instance-segmentation'
EXPECTED_IMAGE_COUNT = 223
EXPECTED_CLASS_NAMES = {'BG', 'WSSV'}
ROBOFLOW_VERSION = 1
ROBOFLOW_FORMAT = 'yolo26'

if importlib.util.find_spec('roboflow') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow'])

from roboflow import Roboflow

def get_roboflow_api_key():
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
        if value:
            return value.strip()
    except Exception:
        pass
    return os.environ.get('ROBOFLOW_API_KEY', '').strip()

api_key = get_roboflow_api_key()
if not api_key:
    raise RuntimeError(
        'Missing ROBOFLOW_API_KEY. Add it to Kaggle Secrets or the environment; do not paste it into the notebook.'
    )

rf = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
if project.id != EXPECTED_PROJECT_ID:
    raise RuntimeError(f'Wrong Roboflow project loaded: expected={EXPECTED_PROJECT_ID}, got={project.id}')
if project.type != EXPECTED_PROJECT_TYPE:
    raise RuntimeError(f'Wrong project type: expected={EXPECTED_PROJECT_TYPE}, got={project.type}')
if project.images != EXPECTED_IMAGE_COUNT:
    raise RuntimeError(f'Wrong project image count: expected={EXPECTED_IMAGE_COUNT}, got={project.images}')
if set(project.classes) != EXPECTED_CLASS_NAMES:
    raise RuntimeError(f'Wrong project classes: expected={EXPECTED_CLASS_NAMES}, got={set(project.classes)}')

dataset = project.version(ROBOFLOW_VERSION).download(ROBOFLOW_FORMAT)
DATASET_LOCATION = Path(dataset.location)
RAW_YOLO_DATASET_PATH = DATASET_LOCATION
print('Downloaded dataset location:', DATASET_LOCATION)


In [ ]:
# Leakage-aware derived split for Mr_TU_ShrimpDiseaseSeg_Dat-Copy.
#
# This cell never moves files out of the Roboflow export. It materializes a
# derived dataset using hard links where possible (copies otherwise), then
# writes a manifest and audit files alongside its data.yaml. Its grouping is:
#   1) filename convention group: disease::shrimp_id, when available;
#   2) verified visual near-duplicate cluster, including duplicates whose
#      filenames do not match the convention;
#   3) a singleton for everything else.
# Every connected group is assigned to exactly one split.

import csv
import hashlib
import importlib.util
import json
import os
import random
import re
import shutil
import subprocess
import sys
from collections import Counter, defaultdict
from pathlib import Path

if importlib.util.find_spec('cv2') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'])
if importlib.util.find_spec('yaml') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'])

import cv2
import numpy as np
import yaml

SEED = 42
SPLIT_RATIOS = {'train': 0.80, 'valid': 0.10, 'test': 0.10}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
FILENAME_GROUP_PATTERN = re.compile(
    r'^(?P<disease>Healthy|BG|WSSV_BG|WSSV)-(?P<shrimp_id>.+)-img-(?P<img_num>\d+)$',
    re.IGNORECASE,
)

# These thresholds were chosen for the currently supplied 1,452-image export.
# Candidate hashes are only a fast filter; the MAE + global SSIM check decides
# whether two images form the same visual duplicate component.
PHASH_MAX_DISTANCE = 8
AHASH_MAX_DISTANCE = 3
DHASH_MAX_DISTANCE = 4
MAX_NORMALIZED_MAE = 0.0065
MIN_GLOBAL_SSIM = 0.985
VERIFY_RESOLUTION = 256

# Set to True only to discard and recreate the generated dataset below. This
# target is derived data under WORK_ROOT, never the Roboflow download itself.
REBUILD_PREPARED_SPLIT = False
PREPARED_DIR = WORK_ROOT / 'mr_tu_shrimpdiseaseseg_dat_copy_v1_yolo26_grouped_near_duplicate_split'


def find_dataset_root(location: Path) -> Path:
    location = Path(location)
    if (location / 'data.yaml').is_file():
        return location
    candidates = sorted(location.rglob('data.yaml'))
    if not candidates:
        raise FileNotFoundError(f'No data.yaml found below {location}')
    return candidates[0].parent


def normalize_roboflow_stem(stem: str) -> str:
    stem = re.sub(r'_(jpg|jpeg|png|bmp|webp)\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    return re.sub(r'\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)


def filename_group_key(image_name: str) -> tuple[str | None, str | None]:
    stem = normalize_roboflow_stem(Path(image_name).stem)
    match = FILENAME_GROUP_PATTERN.match(stem)
    if not match:
        return None, None
    disease = match.group('disease').lower()
    shrimp_id = match.group('shrimp_id')
    return f'filename::{disease}::{shrimp_id}', f'{disease}::{shrimp_id}'


def label_path_for(image_path: Path) -> Path:
    labels_dir = image_path.parent.parent / 'labels'
    direct = labels_dir / f'{image_path.stem}.txt'
    if direct.exists():
        return direct
    matches = sorted(labels_dir.glob(f'{image_path.stem}*.txt'))
    return matches[0] if matches else direct


def prepared_image_name(source_name: str) -> str:
    # Roboflow exports can contain very long scraped-web filenames. A compact,
    # deterministic destination name keeps the derived data usable on Windows,
    # Kaggle and Colab while the manifest preserves the original filename.
    suffix = Path(source_name).suffix.lower()
    digest = hashlib.sha256(source_name.encode('utf-8')).hexdigest()[:20]
    return f'img_{digest}{suffix}'


def parse_class_names(raw_yaml: dict) -> dict[int, str]:
    names = raw_yaml.get('names', {})
    if isinstance(names, list):
        return {index: str(value) for index, value in enumerate(names)}
    return {int(key): str(value) for key, value in names.items()}


def label_ids(label_path: Path, image_name: str, class_names: dict[int, str]) -> tuple[int, ...]:
    if not label_path.exists():
        return ()
    ids = set()
    for line_number, line in enumerate(label_path.read_text(encoding='utf-8').splitlines(), start=1):
        tokens = line.strip().split()
        if not tokens:
            continue
        if len(tokens) < 7 or len(tokens[1:]) % 2 != 0:
            raise ValueError(f'Malformed YOLO segmentation polygon: {image_name}:{line_number}')
        try:
            class_id = int(float(tokens[0]))
            coordinates = [float(value) for value in tokens[1:]]
        except ValueError as error:
            raise ValueError(f'Non-numeric YOLO segmentation label: {image_name}:{line_number}') from error
        if class_id not in class_names:
            raise ValueError(f'Class id {class_id} is absent from data.yaml: {image_name}:{line_number}')
        if any(value < 0.0 or value > 1.0 for value in coordinates):
            raise ValueError(f'Polygon coordinate outside [0, 1]: {image_name}:{line_number}')
        ids.add(class_id)
    return tuple(sorted(ids))


def bits_to_int(bits: np.ndarray) -> int:
    value = 0
    for bit in bits.ravel():
        value = (value << 1) | int(bool(bit))
    return value


def image_hashes(image_path: Path) -> tuple[int, int, int]:
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f'Unreadable image: {image_path}')
    a_small = cv2.resize(image, (8, 8), interpolation=cv2.INTER_AREA)
    a_hash = bits_to_int(a_small >= a_small.mean())
    d_small = cv2.resize(image, (9, 8), interpolation=cv2.INTER_AREA)
    d_hash = bits_to_int(d_small[:, 1:] >= d_small[:, :-1])
    p_small = cv2.resize(image, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    dct = cv2.dct(p_small)[:8, :8]
    median = np.median(dct.ravel()[1:])
    p_hash = bits_to_int(dct >= median)
    return p_hash, a_hash, d_hash


def hamming_distance(left: int, right: int) -> int:
    return (left ^ right).bit_count()


def normalized_image(image_path: Path) -> np.ndarray:
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f'Unreadable image: {image_path}')
    return cv2.resize(image, (VERIFY_RESOLUTION, VERIFY_RESOLUTION), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0


def global_ssim(left: np.ndarray, right: np.ndarray) -> float:
    mean_left, mean_right = float(left.mean()), float(right.mean())
    variance_left, variance_right = float(left.var()), float(right.var())
    covariance = float(((left - mean_left) * (right - mean_right)).mean())
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    return ((2 * mean_left * mean_right + c1) * (2 * covariance + c2)) / (
        (mean_left ** 2 + mean_right ** 2 + c1) * (variance_left + variance_right + c2)
    )


class UnionFind:
    def __init__(self, values):
        self.parent = {value: value for value in values}

    def find(self, value):
        parent = self.parent[value]
        if parent != value:
            self.parent[value] = self.find(parent)
        return self.parent[value]

    def union(self, left, right):
        left_root, right_root = self.find(left), self.find(right)
        if left_root != right_root:
            self.parent[right_root] = left_root


def verified_near_duplicate_edges(records: list[dict]) -> list[dict]:
    cached_normalized = {}
    for record in records:
        record['p_hash'], record['a_hash'], record['d_hash'] = image_hashes(record['image_path'])

    edges = []
    for left_index, left in enumerate(records):
        for right in records[left_index + 1:]:
            p_distance = hamming_distance(left['p_hash'], right['p_hash'])
            a_distance = hamming_distance(left['a_hash'], right['a_hash'])
            d_distance = hamming_distance(left['d_hash'], right['d_hash'])
            if not (
                p_distance <= PHASH_MAX_DISTANCE
                and a_distance <= AHASH_MAX_DISTANCE
                and d_distance <= DHASH_MAX_DISTANCE
            ):
                continue
            left_image = cached_normalized.setdefault(left['name'], normalized_image(left['image_path']))
            right_image = cached_normalized.setdefault(right['name'], normalized_image(right['image_path']))
            mae = float(np.abs(left_image - right_image).mean())
            ssim = global_ssim(left_image, right_image)
            if mae <= MAX_NORMALIZED_MAE and ssim >= MIN_GLOBAL_SSIM:
                edges.append({
                    'left': left['name'],
                    'right': right['name'],
                    'p_hash_distance': p_distance,
                    'a_hash_distance': a_distance,
                    'd_hash_distance': d_distance,
                    'normalized_mae': round(mae, 7),
                    'global_ssim': round(ssim, 7),
                    'label_conflict': left['label_ids'] != right['label_ids'],
                })
    return edges


def target_group_counts(number_of_groups: int) -> dict[str, int]:
    if number_of_groups == 0:
        return {split: 0 for split in SPLIT_RATIOS}
    if number_of_groups == 1:
        return {'train': 1, 'valid': 0, 'test': 0}
    if number_of_groups == 2:
        return {'train': 1, 'valid': 0, 'test': 1}
    valid = max(1, int(round(number_of_groups * SPLIT_RATIOS['valid'])))
    test = max(1, int(round(number_of_groups * SPLIT_RATIOS['test'])))
    train = number_of_groups - valid - test
    if train < 1:
        train = 1
        if valid >= test:
            valid -= 1
        else:
            test -= 1
    return {'train': train, 'valid': valid, 'test': test}


def split_grouped_records(groups: list[dict], seed: int) -> dict[str, str]:
    by_stratum = defaultdict(list)
    for group in groups:
        by_stratum[group['label_stratum']].append(group)

    assignments = {}
    for stratum_index, (stratum, stratum_groups) in enumerate(sorted(by_stratum.items())):
        shuffled = sorted(stratum_groups, key=lambda item: item['group_id'])
        random.Random(seed + stratum_index).shuffle(shuffled)
        group_targets = target_group_counts(len(shuffled))
        image_targets = {
            split: sum(len(group['members']) for group in shuffled) * SPLIT_RATIOS[split]
            for split in SPLIT_RATIOS
        }
        group_counts = Counter()
        image_counts = Counter()
        for group in sorted(shuffled, key=lambda item: (-len(item['members']), item['group_id'])):
            available = [split for split in SPLIT_RATIOS if group_counts[split] < group_targets[split]]
            if not available:
                available = list(SPLIT_RATIOS)
            split = min(
                available,
                key=lambda candidate: (
                    image_counts[candidate] / max(image_targets[candidate], 1.0),
                    group_counts[candidate] / max(group_targets[candidate], 1),
                    candidate,
                ),
            )
            group_counts[split] += 1
            image_counts[split] += len(group['members'])
            for record in group['members']:
                assignments[record['name']] = split
        print(
            f'{stratum}: groups={len(shuffled)} -> '
            + ', '.join(f'{split}={group_counts[split]} groups/{image_counts[split]} images' for split in SPLIT_RATIOS)
        )
    return assignments


def link_or_copy(source: Path, destination: Path):
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.link(source, destination)
    except OSError:
        shutil.copy2(source, destination)


def safe_reset_prepared_dir(path: Path):
    if path.resolve().parent != WORK_ROOT.resolve():
        raise RuntimeError(f'Refusing to reset a directory outside WORK_ROOT: {path}')
    if path.name != 'mr_tu_shrimpdiseaseseg_dat_copy_v1_yolo26_grouped_near_duplicate_split':
        raise RuntimeError(f'Refusing to reset an unexpected directory: {path}')
    shutil.rmtree(path)


RAW_DATASET_DIR = find_dataset_root(Path(DATASET_LOCATION))
raw_yaml_path = RAW_DATASET_DIR / 'data.yaml'
raw_yaml = yaml.safe_load(raw_yaml_path.read_text(encoding='utf-8')) or {}
CLASS_NAMES = parse_class_names(raw_yaml)
if not CLASS_NAMES:
    raise ValueError('data.yaml contains no class names.')

def raw_dataset_fingerprint(dataset_root: Path) -> tuple[str, int]:
    digest = hashlib.sha256()
    digest.update(raw_yaml_path.read_bytes())
    image_count = 0
    for split_name in ('train', 'valid', 'test'):
        image_dir = dataset_root / split_name / 'images'
        if not image_dir.exists():
            continue
        for image_path in sorted(path for path in image_dir.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS):
            label_path = label_path_for(image_path)
            label_digest = hashlib.sha256(label_path.read_bytes()).hexdigest() if label_path.exists() else 'missing'
            digest.update(f'{split_name}/{image_path.name}|{image_path.stat().st_size}|{label_digest}\n'.encode('utf-8'))
            image_count += 1
    return digest.hexdigest(), image_count


SOURCE_FINGERPRINT, SOURCE_IMAGE_COUNT = raw_dataset_fingerprint(RAW_DATASET_DIR)
expected_image_count = globals().get('EXPECTED_IMAGE_COUNT')
if expected_image_count is not None and SOURCE_IMAGE_COUNT != expected_image_count:
    raise RuntimeError(
        f'Wrong downloaded export: expected {expected_image_count} images, found {SOURCE_IMAGE_COUNT}. '
        'Stop and verify the Roboflow project/version before splitting.'
    )

existing_manifest = PREPARED_DIR / 'split_manifest.csv'
existing_yaml = PREPARED_DIR / 'data.yaml'
existing_audit = PREPARED_DIR / 'split_audit.json'
if PREPARED_DIR.exists() and not REBUILD_PREPARED_SPLIT:
    if not (existing_manifest.is_file() and existing_yaml.is_file() and existing_audit.is_file()):
        raise RuntimeError(
            f'{PREPARED_DIR} already exists but is incomplete. Set REBUILD_PREPARED_SPLIT=True to recreate it.'
        )
    prior_audit = json.loads(existing_audit.read_text(encoding='utf-8'))
    if prior_audit.get('source_fingerprint') != SOURCE_FINGERPRINT:
        raise RuntimeError(
            'The existing prepared split belongs to a different raw export and will not be reused. '
            'Set REBUILD_PREPARED_SPLIT=True to recreate this derived dataset.'
        )
    base_path = PREPARED_DIR
    data_yaml_path = existing_yaml
    print('Reusing prepared grouped split for the verified current export:', PREPARED_DIR)
    print('Manifest:', existing_manifest)
else:
    if PREPARED_DIR.exists():
        safe_reset_prepared_dir(PREPARED_DIR)

    records = []
    seen_names = set()
    for original_split in ('train', 'valid', 'test'):
        image_dir = RAW_DATASET_DIR / original_split / 'images'
        if not image_dir.exists():
            continue
        for image_path in sorted(image_dir.iterdir()):
            if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            if image_path.name in seen_names:
                raise RuntimeError(f'Duplicate image filename across source splits: {image_path.name}')
            seen_names.add(image_path.name)
            label_path = label_path_for(image_path)
            group_key, parsed_group = filename_group_key(image_path.name)
            records.append({
                'name': image_path.name,
                'image_path': image_path,
                'label_path': label_path,
                'prepared_name': prepared_image_name(image_path.name),
                'original_split': original_split,
                'filename_group': parsed_group or '',
                'initial_group': group_key or f'singleton::{image_path.name}',
                'label_ids': label_ids(label_path, image_path.name, CLASS_NAMES),
            })

    if not records:
        raise RuntimeError(f'No supported images found below {RAW_DATASET_DIR}')
    print(f'Loaded {len(records)} images from raw export: {RAW_DATASET_DIR}')

    duplicate_edges = verified_near_duplicate_edges(records)
    print(f'Verified near-duplicate edges: {len(duplicate_edges)}')

    union_find = UnionFind(record['name'] for record in records)
    first_by_filename_group = {}
    for record in records:
        initial_group = record['initial_group']
        if initial_group in first_by_filename_group:
            union_find.union(first_by_filename_group[initial_group], record['name'])
        else:
            first_by_filename_group[initial_group] = record['name']
    for edge in duplicate_edges:
        union_find.union(edge['left'], edge['right'])

    by_component = defaultdict(list)
    for record in records:
        by_component[union_find.find(record['name'])].append(record)

    grouped_records = []
    for component_root, members in sorted(by_component.items()):
        members = sorted(members, key=lambda item: item['name'])
        unioned_label_ids = tuple(sorted({class_id for item in members for class_id in item['label_ids']}))
        label_stratum = '+'.join(map(str, unioned_label_ids)) if unioned_label_ids else 'healthy_empty'
        label_signatures = {item['label_ids'] for item in members}
        grouped_records.append({
            'group_id': f'component::{members[0]["name"]}',
            'component_root': component_root,
            'members': members,
            'label_stratum': label_stratum,
            'label_conflict': len(label_signatures) > 1,
        })

    assignments = split_grouped_records(grouped_records, SEED)
    if len(assignments) != len(records):
        raise RuntimeError('Not every image received a split assignment.')

    PREPARED_DIR.mkdir(parents=True, exist_ok=True)
    group_for_name = {
        record['name']: group
        for group in grouped_records
        for record in group['members']
    }
    rows = []
    for record in records:
        split = assignments[record['name']]
        image_destination = PREPARED_DIR / split / 'images' / record['prepared_name']
        label_destination = PREPARED_DIR / split / 'labels' / f'{Path(record["prepared_name"]).stem}.txt'
        link_or_copy(record['image_path'], image_destination)
        if record['label_path'].exists():
            link_or_copy(record['label_path'], label_destination)
        else:
            label_destination.parent.mkdir(parents=True, exist_ok=True)
            label_destination.write_text('', encoding='utf-8')
        group = group_for_name[record['name']]
        rows.append({
            'image_name': record['name'],
            'prepared_image_name': record['prepared_name'],
            'source_image': str(record['image_path']),
            'source_label': str(record['label_path']),
            'source_split': record['original_split'],
            'filename_group': record['filename_group'],
            'component_id': group['group_id'],
            'label_stratum': group['label_stratum'],
            'image_label_ids': '+'.join(map(str, record['label_ids'])) or 'healthy_empty',
            'component_label_conflict': group['label_conflict'],
            'split': split,
        })

    with (PREPARED_DIR / 'split_manifest.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(sorted(rows, key=lambda item: item['image_name']))
    with (PREPARED_DIR / 'near_duplicate_edges.json').open('w', encoding='utf-8') as handle:
        json.dump(duplicate_edges, handle, indent=2, ensure_ascii=False)

    split_image_counts = Counter(assignments.values())
    conflict_groups = [group for group in grouped_records if group['label_conflict']]
    audit = {
        'seed': SEED,
        'ratios': SPLIT_RATIOS,
        'raw_dataset': str(RAW_DATASET_DIR),
        'expected_project_id': globals().get('EXPECTED_PROJECT_ID'),
        'source_fingerprint': SOURCE_FINGERPRINT,
        'source_image_count': SOURCE_IMAGE_COUNT,
        'images': len(records),
        'groups': len(grouped_records),
        'filename_grouped_images': sum(bool(record['filename_group']) for record in records),
        'unmatched_filename_images': sum(not bool(record['filename_group']) for record in records),
        'verified_near_duplicate_edges': len(duplicate_edges),
        'label_conflict_groups': len(conflict_groups),
        'split_image_counts': dict(split_image_counts),
        'class_names': CLASS_NAMES,
    }
    (PREPARED_DIR / 'split_audit.json').write_text(json.dumps(audit, indent=2, ensure_ascii=False), encoding='utf-8')

    prepared_yaml = dict(raw_yaml)
    prepared_yaml['train'] = str((PREPARED_DIR / 'train' / 'images').resolve())
    prepared_yaml['val'] = str((PREPARED_DIR / 'valid' / 'images').resolve())
    prepared_yaml['test'] = str((PREPARED_DIR / 'test' / 'images').resolve())
    prepared_yaml['nc'] = len(CLASS_NAMES)
    prepared_yaml['names'] = [CLASS_NAMES[index] for index in sorted(CLASS_NAMES)]
    prepared_yaml['split_policy'] = 'filename groups + verified visual near-duplicate components; group-stratified 80/10/10'
    data_yaml_path = PREPARED_DIR / 'data.yaml'
    data_yaml_path.write_text(yaml.safe_dump(prepared_yaml, sort_keys=False, allow_unicode=True), encoding='utf-8')

    base_path = PREPARED_DIR
    print('Prepared split:', PREPARED_DIR)
    print('Image counts:', dict(sorted(split_image_counts.items())))
    print('Groups with heterogeneous labels (review in split_manifest.csv):', len(conflict_groups))
    print('Audit:', PREPARED_DIR / 'split_audit.json')

base_path = Path(base_path)
data_yaml_path = Path(data_yaml_path)
print('Prepared data.yaml:', data_yaml_path)


In [ ]:
# Kaggle-standalone novel attention modules and safe Ultralytics parser patch.
import importlib.util
import inspect
import math
import subprocess
import sys
from pathlib import Path

PINNED_ULTRALYTICS_REF = "v8.4.61"
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else (Path("/content") if Path("/content").exists() else Path.cwd())

import torch
import torch.nn as nn
import torch.nn.functional as F

VENDORED_ULTRALYTICS = WORK_ROOT / 'ultralytics-v8.4.61'
if not VENDORED_ULTRALYTICS.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', PINNED_ULTRALYTICS_REF, 'https://github.com/ultralytics/ultralytics.git', str(VENDORED_ULTRALYTICS)])
if str(VENDORED_ULTRALYTICS) not in sys.path:
    sys.path.insert(0, str(VENDORED_ULTRALYTICS))

loaded_ultralytics = sys.modules.get('ultralytics')
if loaded_ultralytics is not None:
    loaded_path = str(getattr(loaded_ultralytics, '__file__', '')).replace('\\', '/')
    expected_path = str(VENDORED_ULTRALYTICS).replace('\\', '/')
    if expected_path not in loaded_path:
        for module_name in list(sys.modules):
            if module_name == 'ultralytics' or module_name.startswith('ultralytics.'):
                del sys.modules[module_name]


class DPCAGate(nn.Module):
    """Dual-Polarity Contrast Attention; shape-preserving and independent of prior attention blocks."""
    def __init__(self, c1, local_kernel=5, gamma_init=0.0):
        super().__init__()
        c1, local_kernel = int(c1), int(local_kernel) | 1
        pad = local_kernel // 2
        self.local = nn.AvgPool2d(local_kernel, stride=1, padding=pad)
        self.selector = nn.Conv2d(2 * c1, c1, 1, groups=c1, bias=True)
        self.refine = nn.Conv2d(c1, c1, 3, padding=1, groups=c1, bias=False)
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))
    def forward(self, x):
        deviation = x - self.local(x)
        bright, dark = F.relu(deviation), F.relu(-deviation)
        choose_bright = torch.sigmoid(self.selector(torch.cat((bright, dark), dim=1)))
        evidence = choose_bright * bright + (1.0 - choose_bright) * dark
        gate = torch.sigmoid(self.refine(evidence))
        return x + self.gamma * x * (gate - 0.5)


class LACAGate(nn.Module):
    """Local Agreement Contrast Attention; prevents isolated contrast from dominating attention."""
    def __init__(self, c1, context_kernel=5, agreement_kernel=3, tau=0.25, gamma_init=0.0):
        super().__init__()
        c1 = int(c1)
        context_kernel, agreement_kernel = int(context_kernel) | 1, int(agreement_kernel) | 1
        self.context = nn.AvgPool2d(context_kernel, 1, context_kernel // 2)
        self.agreement = nn.AvgPool2d(agreement_kernel, 1, agreement_kernel // 2)
        self.tau = nn.Parameter(torch.tensor(float(tau)))
        self.refine = nn.Conv2d(c1, c1, 3, padding=1, groups=c1, bias=False)
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))
    def forward(self, x):
        contrast = (x - self.context(x)).abs()
        neighbourhood = self.agreement(contrast)
        temperature = self.tau.abs().clamp_min(1e-4)
        consensus = torch.exp(-(contrast - neighbourhood).abs() / temperature)
        gate = torch.sigmoid(self.refine(contrast * consensus))
        return x + self.gamma * x * (gate - 0.5)


class ODAAGate(nn.Module):
    """Orientation-Diversity Attention using four learned depthwise directional responses."""
    def __init__(self, c1, temperature=1.0, gamma_init=0.0):
        super().__init__()
        c1 = int(c1)
        self.horizontal = nn.Conv2d(c1, c1, (1, 3), padding=(0, 1), groups=c1, bias=False)
        self.vertical = nn.Conv2d(c1, c1, (3, 1), padding=(1, 0), groups=c1, bias=False)
        self.diag_a = nn.Conv2d(c1, c1, 3, padding=1, groups=c1, bias=False)
        self.diag_b = nn.Conv2d(c1, c1, 3, padding=1, groups=c1, bias=False)
        self.temperature = nn.Parameter(torch.tensor(float(temperature)))
        self.refine = nn.Conv2d(c1, c1, 1, bias=True)
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))
    def forward(self, x):
        responses = torch.stack((self.horizontal(x).abs(), self.vertical(x).abs(), self.diag_a(x).abs(), self.diag_b(x).abs()), dim=1)
        temp = self.temperature.abs().clamp_min(1e-4)
        p = torch.softmax(responses / temp, dim=1)
        diversity = -(p * torch.log(p.clamp_min(1e-8))).sum(dim=1) / math.log(4.0)
        gate = torch.sigmoid(self.refine(diversity))
        return x + self.gamma * x * (gate - 0.5)


class SREAGate(nn.Module):
    """Scale-Residual Evidence Attention; independent fine/coarse residual agreement."""
    def __init__(self, c1, fine_kernel=3, coarse_kernel=7, gamma_init=0.0):
        super().__init__()
        c1 = int(c1)
        fine_kernel, coarse_kernel = int(fine_kernel) | 1, int(coarse_kernel) | 1
        self.fine = nn.AvgPool2d(fine_kernel, 1, fine_kernel // 2)
        self.coarse = nn.AvgPool2d(coarse_kernel, 1, coarse_kernel // 2)
        self.refine = nn.Conv2d(c1, c1, 3, padding=1, groups=c1, bias=False)
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))
    def forward(self, x):
        fine_context = self.fine(x)
        fine_residual = (x - fine_context).abs()
        coarse_residual = (fine_context - self.coarse(x)).abs()
        evidence = fine_residual * torch.sigmoid(coarse_residual)
        gate = torch.sigmoid(self.refine(evidence))
        return x + self.gamma * x * (gate - 0.5)


NOVEL_ATTENTION_MODULES = (DPCAGate, LACAGate, ODAAGate, SREAGate)

def register_novel_attention():
    import ultralytics.nn.modules as nn_modules
    import ultralytics.nn.modules.conv as conv_module
    import ultralytics.nn.tasks as tasks_module
    for cls in NOVEL_ATTENTION_MODULES:
        setattr(tasks_module, cls.__name__, cls)
        setattr(nn_modules, cls.__name__, cls)
        setattr(conv_module, cls.__name__, cls)
    tasks_module.NOVEL_ATTENTION_MODULES = NOVEL_ATTENTION_MODULES
    if getattr(tasks_module, '_novel_shrimp_attention_patched', False):
        return
    source = inspect.getsource(tasks_module.parse_model)
    branch = """        elif m in NOVEL_ATTENTION_MODULES:
            c1 = ch[f]
            c2 = c1
            args = [c1, *args]
"""
    if 'elif m in NOVEL_ATTENTION_MODULES:' not in source:
        marker = '        elif m in frozenset(\n            {\n                Detect,'
        if marker not in source:
            raise RuntimeError('Unsupported Ultralytics parse_model layout; cannot safely register custom attention.')
        source = source.replace(marker, branch + marker, 1)
    exec(compile(source, '<novel_attention_parse_model>', 'exec'), tasks_module.__dict__)
    tasks_module._novel_shrimp_attention_patched = True

register_novel_attention()
print('Registered:', ', '.join(cls.__name__ for cls in NOVEL_ATTENTION_MODULES))


In [ ]:
# Create a YOLO11n-seg YAML with this module at P3, P4, and P5, then verify build/shape.
from pathlib import Path

if "WORK_ROOT" not in globals():
    WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else (Path("/content") if Path("/content").exists() else Path.cwd())
import torch
from ultralytics import YOLO

YAML_CONTENT = """nc: 2
scales:
  n: [0.50, 0.25, 1024]
backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]
head:
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]
  - [16, 1, DPCAGate, []]
  - [19, 1, DPCAGate, []]
  - [22, 1, DPCAGate, []]
  - [[23, 24, 25], 1, Segment, [nc, 32, 256]]
"""

WORKING_ROOT = WORK_ROOT
EXPERIMENT_NAME = 'dpca_p3p4p5_strong_mrtu_grouped_near_duplicate_split'
MODULE_FOLDER_NAME = EXPERIMENT_NAME
yaml_dir = WORKING_ROOT / 'new_research_attention' / 'dpca_dual_polarity_contrast' / 'generated_yamls'
yaml_dir.mkdir(parents=True, exist_ok=True)
MODEL_YAML = str(yaml_dir / f"{EXPERIMENT_NAME}.yaml")
Path(MODEL_YAML).write_text(YAML_CONTENT, encoding='utf-8')

def _assert_module_shape(module_cls, channels):
    for h, w in ((80, 80), (40, 40), (20, 20), (37, 53)):
        x = torch.randn(2, channels, h, w)
        y = module_cls(channels)(x)
        assert y.shape == x.shape, f'{module_cls.__name__}: {tuple(x.shape)} -> {tuple(y.shape)}'
        assert torch.isfinite(y).all(), f'{module_cls.__name__} produced NaN/Inf'

for module_cls in NOVEL_ATTENTION_MODULES:
    for channels in (64, 128, 256):
        _assert_module_shape(module_cls, channels)
print('Standalone module shape checks passed.')

register_novel_attention()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
yolo = YOLO(MODEL_YAML)
yolo.model.to(device).eval()
with torch.no_grad():
    output = yolo.model(torch.zeros(1, 3, 640, 640, device=device))
assert any(isinstance(m, DPCAGate) for m in yolo.model.modules()), 'Custom module missing from parsed model'
print('YOLO build/forward PASS:', MODEL_YAML)


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('ultralytics') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'])

from ultralytics import YOLO
import os
from pathlib import Path

if "base_path" not in globals() or "data_yaml_path" not in globals():
    raise RuntimeError("Run the grouped near-duplicate split cell before configuring the model.")
base_path = str(Path(base_path))
data_yaml_path = str(Path(data_yaml_path))
data_yaml_path = os.path.join(base_path, 'data.yaml')

# Keep the model name and run name tied together so reports are not mislabeled.
# Comment in/out models here for the baseline run.
YOLO_MODELS = [MODEL_YAML]

YOLO_MODEL = MODEL_YAML
MODEL_STEM = EXPERIMENT_NAME
RUN_BASE_NAME = EXPERIMENT_NAME

# Load once here as a smoke check. Training cells instantiate fresh models per experiment.
model = YOLO(YOLO_MODEL)
print('Configured segmentation models:')
for configured_model in YOLO_MODELS:
    print(f'  - {configured_model}')
print(f'Run name prefix: {RUN_BASE_NAME}')


In [ ]:
from pathlib import Path

if "WORK_ROOT" not in globals():
    WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else (Path("/content") if Path("/content").exists() else Path.cwd())
import yaml

if "base_path" not in globals() or "data_yaml_path" not in globals():
    raise RuntimeError("Run the grouped near-duplicate split cell before this validation cell.")
if "data_yaml_path" not in globals():
    data_yaml_path = str(Path(base_path) / "data.yaml")

config_path = Path(data_yaml_path)
if not config_path.is_file():
    raise FileNotFoundError(f"Prepared data.yaml not found: {config_path}")
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))

for split_key, folder in (("train", "train"), ("val", "valid"), ("test", "test")):
    image_dir = Path(base_path) / folder / "images"
    label_dir = Path(base_path) / folder / "labels"
    images = [path for path in image_dir.iterdir() if path.is_file()]
    labels = list(label_dir.glob("*.txt"))
    if not images:
        raise RuntimeError(f"{split_key} split is empty.")
    if len(images) != len(labels):
        raise RuntimeError(
            f"{split_key} image/label mismatch: {len(images)} images, {len(labels)} labels."
        )
    print(f"{split_key}: {len(images)} images, {len(labels)} labels")
print("Verified names:", config.get("names"))
print("Using:", config_path)


### Exploratory Data Analysis (EDA)
We will analyze the dataset to understand the class distribution and visualize some sample images with their masks.


In [ ]:
import os
import yaml
import matplotlib.pyplot as plt
import cv2
import numpy as np
from collections import Counter
from pathlib import Path

# Load class names from data.yaml. Empty label files are healthy shrimp negatives.
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

class_names = data_config.get('names', [])
HEALTHY_CLASS_NAME = 'healthy'
print(f"Disease mask classes found: {class_names}")
print(f"Empty label files will be treated as: {HEALTHY_CLASS_NAME} shrimp negatives")


def split_label_stats(label_dir):
    instance_counts = Counter()
    labeled_images = 0
    healthy_images = 0
    missing_or_empty = 0
    label_dir = Path(label_dir)
    for label_file in label_dir.glob('*.txt'):
        lines = [line.strip() for line in label_file.read_text().splitlines() if line.strip()]
        if not lines:
            healthy_images += 1
            continue
        labeled_images += 1
        for line in lines:
            class_id = int(float(line.split()[0]))
            instance_counts[class_id] += 1
    return {
        'instance_counts': instance_counts,
        'labeled_images': labeled_images,
        'healthy_images': healthy_images,
        'total_label_files': labeled_images + healthy_images,
    }


stats = {}
for split in ['train', 'valid', 'test']:
    label_dir = os.path.join(base_path, split, 'labels')
    stats[split] = split_label_stats(label_dir)

for split, split_stats in stats.items():
    print(f"\n{split.capitalize()} Split:")
    print(f"  - labeled disease images: {split_stats['labeled_images']}")
    print(f"  - healthy negative images: {split_stats['healthy_images']}")
    for cid, count in split_stats['instance_counts'].items():
        name = class_names[cid] if cid < len(class_names) else f"Unknown({cid})"
        print(f"  - {name}: {count} mask instances")


### Visualizing Class Imbalance
An imbalanced dataset can cause the model to be biased. Let's visualize the distribution across our splits.


In [ ]:
import pandas as pd
import seaborn as sns

instance_plot_data = []
image_plot_data = []
for split, split_stats in stats.items():
    image_plot_data.append({'Split': split, 'Class': HEALTHY_CLASS_NAME, 'Images': split_stats['healthy_images']})
    image_plot_data.append({'Split': split, 'Class': 'diseased_labeled', 'Images': split_stats['labeled_images']})
    for cid, count in split_stats['instance_counts'].items():
        instance_plot_data.append({'Split': split, 'Class': class_names[cid], 'Instances': count})

if instance_plot_data:
    df_instances = pd.DataFrame(instance_plot_data)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_instances, x='Split', y='Instances', hue='Class')
    plt.title('Disease Mask Instance Distribution across Splits')
    plt.show()

if image_plot_data:
    df_images = pd.DataFrame(image_plot_data)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_images, x='Split', y='Images', hue='Class')
    plt.title('Healthy Negative vs Diseased-Labeled Image Counts')
    plt.show()

for split, split_stats in stats.items():
    total_instances = sum(split_stats['instance_counts'].values())
    blackgill_ratio = (split_stats['instance_counts'].get(0, 0) / max(1, total_instances)) * 100
    healthy_ratio = (split_stats['healthy_images'] / max(1, split_stats['total_label_files'])) * 100
    print(f"{split.capitalize()}: {blackgill_ratio:.2f}% blackgill instances; {healthy_ratio:.2f}% healthy negative images")


#### Observations:
1. **High Disease-Class Imbalance**: `blackgill` is a small fraction of disease mask instances.
2. **Healthy Negatives Are Real Data**: Empty label files are healthy shrimp images, not missing annotations.
3. **Evaluation Must Split Responsibilities**:
   - Diseased/labeled images should be evaluated with box/mask mAP and recall.
   - Healthy/empty-label images should be evaluated with false-positive rate and false-positive masks per image.

**Recommendation**: Keep healthy images in training as negative controls, but report healthy false-positive metrics separately instead of mixing them into disease mask mAP interpretation.


### Optional Minority RandAugment Oversampling
The baseline starts with `ENABLE_MINORITY_OVERSAMPLING = False`. Turn it on later to create RandAugment-based minority samples. The implementation uses label-preserving photometric RandAugment operations so segmentation polygons remain valid.


In [ ]:
import cv2
import os
import random
from pathlib import Path
from PIL import Image, ImageEnhance, ImageOps

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

ENABLE_MINORITY_OVERSAMPLING = False
MINORITY_CLASS_ID = 0
MINORITY_OVERSAMPLING_MULTIPLIER = 8
RAND_AUGMENT_NUM_OPS = 2
RAND_AUGMENT_MAGNITUDE = 9
RAND_AUGMENT_MAX_MAGNITUDE = 30

# Geometric RandAugment ops are intentionally excluded here because this is
# segmentation data. Photometric ops preserve the existing polygon labels.
RAND_AUGMENT_OPS = [
    'autocontrast',
    'equalize',
    'solarize',
    'posterize',
    'color',
    'contrast',
    'brightness',
    'sharpness',
]


def find_image_for_label(image_dir, label_file):
    stem = Path(label_file).stem
    for ext in IMAGE_EXTENSIONS:
        candidate = Path(image_dir) / f'{stem}{ext}'
        if candidate.exists():
            return candidate
    return None


def remove_yolo_label_caches(base_path):
    for cache_path in Path(base_path).glob('**/*.cache'):
        cache_path.unlink()
        print(f'Removed stale cache: {cache_path}')


def count_labeled_images(label_dir):
    labeled = 0
    healthy = 0
    for label_path in Path(label_dir).glob('*.txt'):
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        if lines:
            labeled += 1
        else:
            healthy += 1
    return labeled, healthy


def randaugment_level(max_value, signed=False):
    magnitude = RAND_AUGMENT_MAGNITUDE / RAND_AUGMENT_MAX_MAGNITUDE
    value = magnitude * max_value
    if signed and random.random() < 0.5:
        value *= -1
    return value


def apply_randaugment_op(image, op_name):
    if op_name == 'autocontrast':
        return ImageOps.autocontrast(image)
    if op_name == 'equalize':
        return ImageOps.equalize(image)
    if op_name == 'solarize':
        threshold = int(256 - randaugment_level(256))
        return ImageOps.solarize(image, threshold=max(0, min(256, threshold)))
    if op_name == 'posterize':
        bits = int(round(8 - randaugment_level(4)))
        return ImageOps.posterize(image, bits=max(4, min(8, bits)))
    if op_name == 'color':
        factor = 1.0 + randaugment_level(0.9, signed=True)
        return ImageEnhance.Color(image).enhance(max(0.1, factor))
    if op_name == 'contrast':
        factor = 1.0 + randaugment_level(0.9, signed=True)
        return ImageEnhance.Contrast(image).enhance(max(0.1, factor))
    if op_name == 'brightness':
        factor = 1.0 + randaugment_level(0.9, signed=True)
        return ImageEnhance.Brightness(image).enhance(max(0.1, factor))
    if op_name == 'sharpness':
        factor = 1.0 + randaugment_level(0.9, signed=True)
        return ImageEnhance.Sharpness(image).enhance(max(0.1, factor))
    raise ValueError(f'Unsupported RandAugment op: {op_name}')


def apply_label_preserving_randaugment(image):
    image = image.convert('RGB')
    ops = random.choices(RAND_AUGMENT_OPS, k=RAND_AUGMENT_NUM_OPS)
    for op_name in ops:
        image = apply_randaugment_op(image, op_name)
    return image


def augment_minority_class(base_path, class_id_to_target=0, multiplier=5):
    train_img_dir = os.path.join(base_path, 'train', 'images')
    train_lbl_dir = os.path.join(base_path, 'train', 'labels')

    label_files = [f for f in os.listdir(train_lbl_dir) if f.endswith('.txt')]
    augmented_count = 0
    skipped_count = 0
    random.seed(SEED)

    for label_file in label_files:
        if label_file.startswith('randaug_'):
            continue

        label_path = os.path.join(train_lbl_dir, label_file)
        with open(label_path, 'r') as f:
            content = [line.strip() for line in f.readlines() if line.strip()]

        has_target = any(int(line.split()[0]) == class_id_to_target for line in content)
        if not has_target:
            continue

        img_path = find_image_for_label(train_img_dir, label_file)
        if img_path is None:
            skipped_count += 1
            continue

        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            skipped_count += 1
            continue

        for m in range(multiplier):
            aug_img = apply_label_preserving_randaugment(image)
            aug_image_name = f'randaug_{m}_{img_path.stem}.jpg'
            aug_label_name = f'randaug_{m}_{Path(label_file).stem}.txt'
            aug_img.save(os.path.join(train_img_dir, aug_image_name), quality=95)
            with open(os.path.join(train_lbl_dir, aug_label_name), 'w') as f:
                f.write('\n'.join(content) + '\n')
            augmented_count += 1

    remove_yolo_label_caches(base_path)
    labeled, healthy = count_labeled_images(train_lbl_dir)
    target_name = class_names[class_id_to_target] if class_id_to_target < len(class_names) else str(class_id_to_target)
    print(f'Created {augmented_count} RandAugment samples for class {target_name}.')
    print(f'Skipped {skipped_count} source labels due to missing image or invalid labels.')
    print(f'Train labels after augmentation: {labeled} diseased/labeled images, {healthy} healthy negative label files.')


if ENABLE_MINORITY_OVERSAMPLING:
    augment_minority_class(base_path, class_id_to_target=MINORITY_CLASS_ID, multiplier=MINORITY_OVERSAMPLING_MULTIPLIER)
else:
    print('Minority RandAugment oversampling disabled for the clean baseline.')


### Strong-Aug Attention Training and Healthy-Negative Evaluation
Train YOLO with the relabeled dataset, shrimp-level grouped disease-stratified split, and explicit light YOLO augmentation. Mosaic, mixup, copy-paste, erasing, and hidden Albumentations remain disabled. Model selection is validation-only; test metrics are reported after selection.


### Required preflight: paths, segmentation labels, and model shapes

This gate runs immediately before any training. It stops the notebook if a split path is invalid, an image has no matching label file, a polygon label is malformed, or the configured model cannot complete a dummy segmentation forward pass.


In [ ]:
# Mandatory preflight gate. Do not bypass this cell before a real training run.
from pathlib import Path
import math

import torch
import yaml
from ultralytics import YOLO

PREFLIGHT_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def _preflight_image_dir(value, data_config_path):
    """Resolve a YOLO data.yaml split value to its images directory."""
    if isinstance(value, (list, tuple)):
        if len(value) != 1:
            raise ValueError("This notebook requires exactly one image directory per split.")
        value = value[0]
    image_dir = Path(str(value))
    if not image_dir.is_absolute():
        image_dir = data_config_path.parent / image_dir
    return image_dir.resolve()


def _preflight_validate_dataset(data_config_path):
    data_config_path = Path(data_config_path).resolve()
    if not data_config_path.is_file():
        raise FileNotFoundError(f"Prepared data.yaml was not found: {data_config_path}")

    config = yaml.safe_load(data_config_path.read_text(encoding="utf-8")) or {}
    names = config.get("names", {})
    class_count = len(names) if isinstance(names, (list, tuple, dict)) else 0
    if class_count < 1:
        raise ValueError("data.yaml must define at least one class in `names`.")

    summary = {}
    for split_name in ("train", "val", "test"):
        if split_name not in config:
            raise KeyError(f"data.yaml is missing the `{split_name}` split.")
        image_dir = _preflight_image_dir(config[split_name], data_config_path)
        label_dir = image_dir.parent / "labels"
        if not image_dir.is_dir() or not label_dir.is_dir():
            raise FileNotFoundError(
                f"{split_name}: expected images={image_dir} and labels={label_dir}"
            )

        images = sorted(
            path for path in image_dir.iterdir()
            if path.is_file() and path.suffix.lower() in PREFLIGHT_IMAGE_EXTENSIONS
        )
        if not images:
            raise RuntimeError(f"{split_name}: image split is empty: {image_dir}")

        label_files = sorted(path for path in label_dir.glob("*.txt") if path.is_file())
        image_stems = {path.stem for path in images}
        label_stems = {path.stem for path in label_files}
        missing_labels = sorted(image_stems - label_stems)
        orphan_labels = sorted(label_stems - image_stems)
        if missing_labels or orphan_labels:
            raise RuntimeError(
                f"{split_name}: image/label mismatch; "
                f"missing_labels={missing_labels[:5]}, orphan_labels={orphan_labels[:5]}"
            )

        labeled_images = 0
        polygons = 0
        for label_path in label_files:
            lines = [line.strip() for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip()]
            if lines:
                labeled_images += 1
            for line_number, line in enumerate(lines, start=1):
                tokens = line.split()
                if len(tokens) < 7 or (len(tokens) - 1) % 2 != 0:
                    raise ValueError(
                        f"{split_name}: malformed segmentation polygon at "
                        f"{label_path.name}:{line_number}"
                    )
                try:
                    class_id = int(tokens[0])
                    coordinates = [float(token) for token in tokens[1:]]
                except ValueError as exc:
                    raise ValueError(
                        f"{split_name}: non-numeric label at {label_path.name}:{line_number}"
                    ) from exc
                if not 0 <= class_id < class_count:
                    raise ValueError(
                        f"{split_name}: class id {class_id} out of range at "
                        f"{label_path.name}:{line_number}"
                    )
                if any(not math.isfinite(value) or value < 0.0 or value > 1.0 for value in coordinates):
                    raise ValueError(
                        f"{split_name}: polygon coordinate outside [0, 1] at "
                        f"{label_path.name}:{line_number}"
                    )
                polygons += 1
        summary[split_name] = {
            "images": len(images),
            "empty_labels": len(images) - labeled_images,
            "polygons": polygons,
        }
    return summary


def _preflight_flatten_tensors(value):
    if torch.is_tensor(value):
        yield value
    elif isinstance(value, (tuple, list)):
        for item in value:
            yield from _preflight_flatten_tensors(item)
    elif isinstance(value, dict):
        for item in value.values():
            yield from _preflight_flatten_tensors(item)


def run_notebook_preflight(model_specs, expected_attention=None, dummy_imgsz=128):
    """Fail before training on a bad dataset path, label, model build, or tensor shape."""
    if dummy_imgsz % 32:
        raise ValueError("dummy_imgsz must be divisible by 32 for YOLO segmentation.")
    dataset_summary = _preflight_validate_dataset(data_yaml_path)
    print("Dataset preflight PASS:", dataset_summary)

    checked_specs = []
    for model_spec in dict.fromkeys(str(spec) for spec in model_specs):
        model = YOLO(model_spec)
        module = model.model.eval()
        attention_counts = {}
        attention_shape_errors = []
        attention_handles = []
        shape_preserving_names = {
            "CoTEBoundaryLiteGate", "DPCAGate", "LargeKernelAttention", "CoordAtt", "SimAM"
        }

        def _attention_shape_hook(layer_name):
            def hook(_module, inputs, output):
                input_tensor = inputs[0] if inputs else None
                if not torch.is_tensor(input_tensor) or not torch.is_tensor(output):
                    attention_shape_errors.append(f"{layer_name}: non-tensor attention input/output")
                elif input_tensor.ndim != 4 or output.ndim != 4 or tuple(input_tensor.shape) != tuple(output.shape):
                    attention_shape_errors.append(
                        f"{layer_name}: expected shape-preserving BCHW, got "
                        f"{tuple(input_tensor.shape)} -> {tuple(output.shape)}"
                    )
            return hook

        for layer in module.modules():
            layer_name = layer.__class__.__name__
            if layer_name in shape_preserving_names:
                attention_counts[layer_name] = attention_counts.get(layer_name, 0) + 1
                attention_handles.append(layer.register_forward_hook(_attention_shape_hook(layer_name)))
        try:
            device = next(module.parameters()).device
        except StopIteration:
            device = torch.device("cpu")
        dummy = torch.zeros(1, 3, dummy_imgsz, dummy_imgsz, device=device)
        with torch.inference_mode():
            output = module(dummy)
        for handle in attention_handles:
            handle.remove()
        if attention_shape_errors:
            raise RuntimeError(f"{model_spec}: attention shape check failed: {attention_shape_errors}")
        if expected_attention is not None:
            observed_attention = {name: attention_counts.get(name, 0) for name in expected_attention}
            if observed_attention != expected_attention:
                raise RuntimeError(
                    f"{model_spec}: attention module count mismatch; "
                    f"expected={expected_attention}, observed={observed_attention}"
                )
        tensors = list(_preflight_flatten_tensors(output))
        if not tensors:
            raise RuntimeError(f"{model_spec}: model forward returned no tensors.")
        if any(tensor.numel() == 0 or not torch.isfinite(tensor).all().item() for tensor in tensors):
            raise FloatingPointError(f"{model_spec}: model forward produced empty or non-finite tensors.")
        checked_specs.append(model_spec)
        del model, module, dummy, output, tensors
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("Model build/shape preflight PASS:", checked_specs)
    return dataset_summary


In [ ]:
import csv
import gc
import math
import shutil
import time
from pathlib import Path

if "WORK_ROOT" not in globals():
    WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else (Path("/content") if Path("/content").exists() else Path.cwd())

import cv2
import pandas as pd
import yaml
from IPython.display import display

WORKING_ROOT = WORK_ROOT
EXPERIMENT_ROOT = WORKING_ROOT / 'new_research_attention' / MODULE_FOLDER_NAME / 'outputs' / EXPERIMENT_NAME
RUNS_DIR = WORKING_ROOT / 'runs' / 'new_research_attention'
REPORT_DIR = EXPERIMENT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

SMOKE_RUN = False  # Set True for a quick Kaggle smoke training pass.
TRAIN_IMGSZ = 320 if SMOKE_RUN else 640
TRAIN_EPOCHS = 1 if SMOKE_RUN else 100
TRAIN_BATCH = 8 if SMOKE_RUN else 16
TRAIN_PATIENCE = 1 if SMOKE_RUN else 30
RUN_TEST_EVALUATION = not SMOKE_RUN

COUNT_PENALTY_WEIGHT = 0.05
DISEASE_MISS_PENALTY_WEIGHT = 0.15
HEALTHY_FP_PENALTY_WEIGHT = 0.10
PREDICT_CONF_FOR_COUNT = 0.25

LIGHT_AUG_TRAIN_ARGS = {
    'auto_augment': None, 'erasing': 0.0, 'mosaic': 0.0, 'mixup': 0.0, 'cutmix': 0.0,
    'copy_paste': 0.0, 'fliplr': 0.5, 'flipud': 0.0, 'hsv_h': 0.01, 'hsv_s': 0.35,
    'hsv_v': 0.20, 'degrees': 0.0, 'translate': 0.05, 'scale': 0.20, 'shear': 0.0,
    'perspective': 0.0, 'multi_scale': 0.0, 'bgr': 0.0,
}
STRONG_AUG_TRAIN_ARGS = {
    'auto_augment': None, 'erasing': 0.15, 'mosaic': 0.0, 'mixup': 0.0, 'cutmix': 0.0,
    'copy_paste': 0.0, 'fliplr': 0.5, 'flipud': 0.3, 'hsv_h': 0.05, 'hsv_s': 0.5,
    'hsv_v': 0.4, 'degrees': 10.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0,
    'perspective': 0.0, 'multi_scale': 0.0, 'bgr': 0.0,
}

EXPERIMENTS = [
    {'key': 'dpca_dual_polarity_contrast_light', 'name': 'DPCA — Dual-Polarity Contrast Attention + light baseline augmentation', 'policy': 'light', 'train_args': LIGHT_AUG_TRAIN_ARGS, 'apply_clahe': False},
    {'key': 'dpca_dual_polarity_contrast_strong', 'name': 'DPCA — Dual-Polarity Contrast Attention + strong SimAM+CA-matched augmentation', 'policy': 'strong', 'train_args': STRONG_AUG_TRAIN_ARGS, 'apply_clahe': False},
]



def disable_ultralytics_albumentations():
    """Disable Ultralytics' tiny default Albumentations transforms for a true bare baseline."""
    try:
        import ultralytics.data.augment as yolo_augment
    except Exception as exc:
        print(f'Could not patch Ultralytics Albumentations hook: {exc}')
        return

    class NoOpAlbumentations:
        contains_spatial = False

        def __init__(self, *args, **kwargs):
            self.transform = None

        def __call__(self, labels):
            return labels

    yolo_augment.Albumentations = NoOpAlbumentations
    print('Ultralytics Albumentations hook disabled for this run.')


def remove_yolo_label_caches(root):
    for cache_path in Path(root).glob('**/*.cache'):
        cache_path.unlink()
        print(f'Removed stale cache: {cache_path}')


def apply_clahe_to_dataset(image_dir):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    image_files = []
    for ext in IMAGE_EXTENSIONS:
        image_files.extend(Path(image_dir).glob(f'*{ext}'))

    print(f'Applying CLAHE to {len(image_files)} images in {image_dir}...')
    for img_path in image_files:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        l2 = clahe.apply(l)
        enhanced = cv2.cvtColor(cv2.merge((l2, a, b)), cv2.COLOR_LAB2BGR)
        cv2.imwrite(str(img_path), enhanced)


def count_labeled_images(label_dir):
    labeled = 0
    healthy = 0
    instances = 0
    for label_path in Path(label_dir).glob('*.txt'):
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        if lines:
            labeled += 1
            instances += len(lines)
        else:
            healthy += 1
    return {'labeled_images': labeled, 'healthy_images': healthy, 'instances': instances}


def write_data_yaml(dataset_dir, yaml_path, val_dir='valid', test_dir='test'):
    dataset_dir = Path(dataset_dir).resolve()
    with open(data_yaml_path, 'r') as f:
        content = yaml.safe_load(f)
    content['train'] = str(Path(dataset_dir) / 'train' / 'images')
    content['val'] = str(Path(dataset_dir) / val_dir / 'images')
    content['test'] = str(Path(dataset_dir) / test_dir / 'images')
    with open(yaml_path, 'w') as f:
        yaml.safe_dump(content, f, sort_keys=False)
    return yaml_path


def copy_dataset_for_experiment(exp_key):
    src = Path(base_path)
    dst = EXPERIMENT_ROOT / exp_key / 'dataset'
    if dst.exists():
        shutil.rmtree(dst)
    ignore = shutil.ignore_patterns('runs', '*.cache', '.clahe_applied')
    shutil.copytree(src, dst, ignore=ignore)
    remove_yolo_label_caches(dst)
    return dst


def copy_split_by_label_state(src_dataset, dst_dataset, split, want_labeled):
    src_images = Path(src_dataset) / split / 'images'
    src_labels = Path(src_dataset) / split / 'labels'
    dst_images = Path(dst_dataset) / split / 'images'
    dst_labels = Path(dst_dataset) / split / 'labels'
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    copied = 0
    for label_path in sorted(src_labels.glob('*.txt')):
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        is_labeled = bool(lines)
        if is_labeled != want_labeled:
            continue
        image_path = find_image_for_label(src_images, label_path.name)
        if image_path is None:
            continue
        shutil.copy2(image_path, dst_images / image_path.name)
        shutil.copy2(label_path, dst_labels / label_path.name)
        copied += 1
    return copied


def make_state_eval_dataset(src_dataset, exp_key, state_name, want_labeled):
    dst = EXPERIMENT_ROOT / exp_key / f'dataset_{state_name}_eval'
    if dst.exists():
        shutil.rmtree(dst)

    for sub in ['images', 'labels']:
        (dst / 'train' / sub).mkdir(parents=True, exist_ok=True)
    copied = {}
    for split in ['valid', 'test']:
        copied[split] = copy_split_by_label_state(src_dataset, dst, split, want_labeled=want_labeled)
    yaml_path = dst / f'data_{state_name}.yaml'
    write_data_yaml(dst, yaml_path)
    print(f'{state_name} eval dataset for {exp_key}: {copied}')
    return dst, yaml_path, copied


def make_labeled_only_eval_dataset(src_dataset, exp_key):
    return make_state_eval_dataset(src_dataset, exp_key, 'labeled_only', want_labeled=True)


def make_healthy_only_eval_dataset(src_dataset, exp_key):
    return make_state_eval_dataset(src_dataset, exp_key, 'healthy_only', want_labeled=False)


def metric_value(metrics, dotted_path, default=float('nan')):
    obj = metrics
    for part in dotted_path.split('.'):
        if not hasattr(obj, part):
            return default
        obj = getattr(obj, part)
    try:
        return float(obj)
    except Exception:
        return default


def count_prediction_errors(model, images_dir, labels_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = []
    for ext in IMAGE_EXTENSIONS:
        image_paths.extend(Path(images_dir).glob(f'*{ext}'))
    image_paths = sorted(image_paths)
    if not image_paths:
        return {
            'images': 0,
            'gt_total': 0,
            'pred_box_total': 0,
            'pred_mask_total': 0,
            'box_count_mae': float('nan'),
            'mask_count_mae': float('nan'),
            'box_count_exact': float('nan'),
            'mask_count_exact': float('nan'),
            'disease_images': 0,
            'disease_box_miss_images': 0,
            'disease_mask_miss_images': 0,
            'disease_box_miss_rate': float('nan'),
            'disease_mask_miss_rate': float('nan'),
        }

    results = model.predict(source=[str(p) for p in image_paths], imgsz=TRAIN_IMGSZ, conf=conf, verbose=False)
    box_errors = []
    mask_errors = []
    box_exact = []
    mask_exact = []
    gt_total = 0
    pred_box_total = 0
    pred_mask_total = 0
    disease_images = 0
    disease_box_miss_images = 0
    disease_mask_miss_images = 0

    for image_path, result in zip(image_paths, results):
        label_path = Path(labels_dir) / f'{image_path.stem}.txt'
        gt_count = 0
        if label_path.exists():
            gt_count = len([line for line in label_path.read_text().splitlines() if line.strip()])
        box_count = len(result.boxes) if result.boxes is not None else 0
        mask_count = len(result.masks) if result.masks is not None else 0
        denom = max(1, gt_count)
        box_errors.append(abs(box_count - gt_count) / denom)
        mask_errors.append(abs(mask_count - gt_count) / denom)
        box_exact.append(float(box_count == gt_count))
        mask_exact.append(float(mask_count == gt_count))
        if gt_count > 0:
            disease_images += 1
            disease_box_miss_images += int(box_count == 0)
            disease_mask_miss_images += int(mask_count == 0)
        gt_total += gt_count
        pred_box_total += box_count
        pred_mask_total += mask_count

    disease_box_miss_rate = disease_box_miss_images / disease_images if disease_images else float('nan')
    disease_mask_miss_rate = disease_mask_miss_images / disease_images if disease_images else float('nan')
    return {
        'images': len(image_paths),
        'gt_total': gt_total,
        'pred_box_total': pred_box_total,
        'pred_mask_total': pred_mask_total,
        'box_count_mae': sum(box_errors) / len(box_errors),
        'mask_count_mae': sum(mask_errors) / len(mask_errors),
        'box_count_exact': sum(box_exact) / len(box_exact),
        'mask_count_exact': sum(mask_exact) / len(mask_exact),
        'disease_images': disease_images,
        'disease_box_miss_images': disease_box_miss_images,
        'disease_mask_miss_images': disease_mask_miss_images,
        'disease_box_miss_rate': disease_box_miss_rate,
        'disease_mask_miss_rate': disease_mask_miss_rate,
    }



def _safe_metric_key(name):
    key = ''.join(ch.lower() if ch.isalnum() else '_' for ch in str(name))
    key = '_'.join(part for part in key.split('_') if part)
    return key or 'class'


def _read_label_class_ids(label_path):
    label_path = Path(label_path)
    if not label_path.exists():
        return []
    class_ids = []
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if not parts:
            continue
        try:
            class_ids.append(int(float(parts[0])))
        except Exception:
            continue
    return class_ids


def _prediction_class_ids(result):
    if result.boxes is None or len(result.boxes) == 0:
        return []
    try:
        return [int(v) for v in result.boxes.cls.detach().cpu().tolist()]
    except Exception:
        return []


def image_level_class_error_summary(model, images_dir, labels_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = []
    for ext in IMAGE_EXTENSIONS:
        image_paths.extend(Path(images_dir).glob(f'*{ext}'))
    image_paths = sorted(image_paths)

    per_class = {
        cid: {
            'class_id': cid,
            'class_name': class_names[cid] if cid < len(class_names) else f'Unknown({cid})',
            'gt_images': 0,
            'pred_images': 0,
            'miss_images': 0,
            'fp_images': 0,
            'wrong_images': 0,
        }
        for cid in range(len(class_names))
    }

    if not image_paths:
        return {
            'images': 0,
            'any_wrong_images': 0,
            'any_wrong_rate': float('nan'),
            'healthy_images': 0,
            'healthy_wrong_images': 0,
            'healthy_wrong_rate': float('nan'),
            'diseased_images': 0,
            'class_rows': list(per_class.values()),
        }

    results = model.predict(source=[str(p) for p in image_paths], imgsz=TRAIN_IMGSZ, conf=conf, verbose=False)
    any_wrong_images = 0
    healthy_images = 0
    healthy_wrong_images = 0
    diseased_images = 0

    for image_path, result in zip(image_paths, results):
        gt_classes = _read_label_class_ids(Path(labels_dir) / f'{image_path.stem}.txt')
        pred_classes = _prediction_class_ids(result)
        gt_set = set(gt_classes)
        pred_set = set(pred_classes)

        if gt_classes:
            diseased_images += 1
        else:
            healthy_images += 1
            if pred_classes:
                healthy_wrong_images += 1

        if gt_set != pred_set:
            any_wrong_images += 1

        for cid, row in per_class.items():
            has_gt = cid in gt_set
            has_pred = cid in pred_set
            if has_gt:
                row['gt_images'] += 1
            if has_pred:
                row['pred_images'] += 1
            if has_gt and not has_pred:
                row['miss_images'] += 1
            if has_pred and not has_gt:
                row['fp_images'] += 1
            if has_gt != has_pred:
                row['wrong_images'] += 1

    return {
        'images': len(image_paths),
        'any_wrong_images': any_wrong_images,
        'any_wrong_rate': any_wrong_images / len(image_paths),
        'healthy_images': healthy_images,
        'healthy_wrong_images': healthy_wrong_images,
        'healthy_wrong_rate': healthy_wrong_images / healthy_images if healthy_images else float('nan'),
        'diseased_images': diseased_images,
        'class_rows': list(per_class.values()),
    }


def flatten_image_level_class_errors(prefix, summary):
    flat = {
        f'{prefix}_image_total': summary['images'],
        f'{prefix}_image_wrong_total': summary['any_wrong_images'],
        f'{prefix}_image_wrong_rate': summary['any_wrong_rate'],
        f'{prefix}_healthy_images': summary['healthy_images'],
        f'{prefix}_healthy_wrong_images': summary['healthy_wrong_images'],
        f'{prefix}_healthy_wrong_rate': summary['healthy_wrong_rate'],
        f'{prefix}_diseased_images': summary['diseased_images'],
    }
    for row in summary['class_rows']:
        key = _safe_metric_key(row['class_name'])
        flat[f'{prefix}_{key}_gt_images'] = row['gt_images']
        flat[f'{prefix}_{key}_pred_images'] = row['pred_images']
        flat[f'{prefix}_{key}_miss_images'] = row['miss_images']
        flat[f'{prefix}_{key}_fp_images'] = row['fp_images']
        flat[f'{prefix}_{key}_wrong_images'] = row['wrong_images']
    return flat


def print_image_level_class_errors(title, summary):
    print(f'\n{title}')
    print(f"  Images: {summary['images']}")
    print(f"  Images with wrong predicted class set: {summary['any_wrong_images']} ({summary['any_wrong_rate']:.4f})")
    print(f"  Healthy images predicted wrong: {summary['healthy_wrong_images']} / {summary['healthy_images']} ({summary['healthy_wrong_rate']:.4f})")
    df = pd.DataFrame(summary['class_rows'])
    if not df.empty:
        display(df[['class_id', 'class_name', 'gt_images', 'pred_images', 'miss_images', 'fp_images', 'wrong_images']])

def healthy_false_positive_summary(model, images_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = []
    for ext in IMAGE_EXTENSIONS:
        image_paths.extend(Path(images_dir).glob(f'*{ext}'))
    image_paths = sorted(image_paths)
    if not image_paths:
        return {
            'healthy_images': 0,
            'healthy_images_with_box_fp': 0,
            'healthy_images_with_mask_fp': 0,
            'healthy_box_fp_rate': float('nan'),
            'healthy_mask_fp_rate': float('nan'),
            'healthy_fp_boxes_total': 0,
            'healthy_fp_masks_total': 0,
            'healthy_fp_boxes_per_image': float('nan'),
            'healthy_fp_masks_per_image': float('nan'),
            'healthy_avg_fp_confidence': float('nan'),
        }

    results = model.predict(source=[str(p) for p in image_paths], imgsz=TRAIN_IMGSZ, conf=conf, verbose=False)
    images_with_box_fp = 0
    images_with_mask_fp = 0
    box_total = 0
    mask_total = 0
    confidences = []

    for result in results:
        box_count = len(result.boxes) if result.boxes is not None else 0
        mask_count = len(result.masks) if result.masks is not None else 0
        if box_count > 0:
            images_with_box_fp += 1
            try:
                confidences.extend([float(v) for v in result.boxes.conf.detach().cpu().tolist()])
            except Exception:
                pass
        if mask_count > 0:
            images_with_mask_fp += 1
        box_total += box_count
        mask_total += mask_count

    n = len(image_paths)
    return {
        'healthy_images': n,
        'healthy_images_with_box_fp': images_with_box_fp,
        'healthy_images_with_mask_fp': images_with_mask_fp,
        'healthy_box_fp_rate': images_with_box_fp / n,
        'healthy_mask_fp_rate': images_with_mask_fp / n,
        'healthy_fp_boxes_total': box_total,
        'healthy_fp_masks_total': mask_total,
        'healthy_fp_boxes_per_image': box_total / n,
        'healthy_fp_masks_per_image': mask_total / n,
        'healthy_avg_fp_confidence': sum(confidences) / len(confidences) if confidences else 0.0,
    }


def healthy_aware_score(labeled_map50, count_summary, healthy_fp_summary):
    count_penalty = COUNT_PENALTY_WEIGHT * count_summary['mask_count_mae']
    disease_miss_penalty = DISEASE_MISS_PENALTY_WEIGHT * count_summary['disease_box_miss_rate']
    healthy_fp_penalty = HEALTHY_FP_PENALTY_WEIGHT * healthy_fp_summary['healthy_mask_fp_rate']
    return labeled_map50 - count_penalty - disease_miss_penalty - healthy_fp_penalty


def read_best_epoch_from_results(run_path):
    results_csv = Path(run_path) / 'results.csv'
    if not results_csv.exists():
        return {}
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    mask_col = 'metrics/mAP50(M)'
    if mask_col not in df.columns:
        return {'epochs_ran': len(df)}
    best_idx = df[mask_col].idxmax()
    first = df.iloc[0]
    best = df.iloc[best_idx]
    last = df.iloc[-1]
    return {
        'epochs_ran': int(len(df)),
        'best_epoch_by_mask_map50': int(best['epoch']) if 'epoch' in df.columns else int(best_idx + 1),
        'first_train_seg_loss': float(first.get('train/seg_loss', float('nan'))),
        'best_val_mask_map50': float(best.get(mask_col, float('nan'))),
        'best_val_mask_map50_95': float(best.get('metrics/mAP50-95(M)', float('nan'))),
        'last_val_mask_map50': float(last.get(mask_col, float('nan'))),
        'last_val_mask_map50_95': float(last.get('metrics/mAP50-95(M)', float('nan'))),
        'last_train_seg_loss': float(last.get('train/seg_loss', float('nan'))),
        'last_val_seg_loss': float(last.get('val/seg_loss', float('nan'))),
        'seg_loss_gap_val_minus_train': float(last.get('val/seg_loss', float('nan')) - last.get('train/seg_loss', float('nan'))),
    }


def run_experiment(exp):
    print('\n' + '#' * 90)
    print(f"Starting experiment: {exp['name']}")
    print('#' * 90)

    dataset_dir = copy_dataset_for_experiment(exp['key'])
    if exp.get('apply_clahe', False):
        for split in ['train', 'valid', 'test']:
            apply_clahe_to_dataset(dataset_dir / split / 'images')

    remove_yolo_label_caches(dataset_dir)
    yaml_path = dataset_dir / 'data.yaml'
    write_data_yaml(dataset_dir, yaml_path)
    labeled_eval_dir, labeled_eval_yaml, _ = make_labeled_only_eval_dataset(dataset_dir, exp['key'])
    healthy_eval_dir, healthy_eval_yaml, _ = make_healthy_only_eval_dataset(dataset_dir, exp['key'])

    split_counts = {}
    for split in ['train', 'valid', 'test']:
        split_counts[split] = count_labeled_images(dataset_dir / split / 'labels')
        print(f'{exp["key"]} {split}: {split_counts[split]}')

    run_name = exp['key'] + ('_smoke' if SMOKE_RUN else '')
    disable_ultralytics_albumentations()
    yolo = YOLO(YOLO_MODEL)
    start = time.time()
    yolo.train(
        data=str(yaml_path),
        task='segment',
        imgsz=TRAIN_IMGSZ,
        epochs=TRAIN_EPOCHS,
        batch=TRAIN_BATCH,
        patience=TRAIN_PATIENCE,
        seed=42,
        project=str(RUNS_DIR),
        name=run_name,
        exist_ok=True,
        pretrained=True,
        plots=not SMOKE_RUN,
        verbose=True,
        **exp['train_args'],
    )
    train_time_min = (time.time() - start) / 60

    run_path = RUNS_DIR / run_name
    best_path = run_path / 'weights' / 'best.pt'
    best_model = YOLO(str(best_path))

    full_val = best_model.val(data=str(yaml_path), split='val', imgsz=TRAIN_IMGSZ, plots=not SMOKE_RUN, verbose=False)
    labeled_val = best_model.val(data=str(labeled_eval_yaml), split='val', imgsz=TRAIN_IMGSZ, plots=False, verbose=False)
    labeled_val_count = count_prediction_errors(
        best_model,
        labeled_eval_dir / 'valid' / 'images',
        labeled_eval_dir / 'valid' / 'labels',
    )
    healthy_val_fp = healthy_false_positive_summary(
        best_model,
        healthy_eval_dir / 'valid' / 'images',
    )
    labeled_val_map50 = metric_value(labeled_val, 'seg.map50')
    val_score = healthy_aware_score(labeled_val_map50, labeled_val_count, healthy_val_fp)

    row = {
        'experiment': exp['key'],
        'name': exp['name'],
        'model': YOLO_MODEL,
        'run_name': run_name,
        'run_path': str(run_path),
        'best_pt': str(best_path),
        'smoke_run': SMOKE_RUN,
        'augmentation_policy': exp['policy'],
        'hidden_albumentations_disabled': True,
        'apply_clahe': exp.get('apply_clahe', False),
        'train_time_min': round(train_time_min, 2),
        'full_val_box_map50': metric_value(full_val, 'box.map50'),
        'full_val_mask_map50': metric_value(full_val, 'seg.map50'),
        'labeled_val_box_map50': metric_value(labeled_val, 'box.map50'),
        'labeled_val_mask_map50': labeled_val_map50,
        'labeled_val_mask_map50_95': metric_value(labeled_val, 'seg.map'),
        'labeled_val_gt_instances': labeled_val_count['gt_total'],
        'labeled_val_pred_boxes': labeled_val_count['pred_box_total'],
        'labeled_val_pred_masks': labeled_val_count['pred_mask_total'],
        'labeled_val_mask_count_mae': labeled_val_count['mask_count_mae'],
        'labeled_val_disease_box_miss_rate': labeled_val_count['disease_box_miss_rate'],
        'labeled_val_disease_mask_miss_rate': labeled_val_count['disease_mask_miss_rate'],
        'healthy_val_images': healthy_val_fp['healthy_images'],
        'healthy_val_mask_fp_rate': healthy_val_fp['healthy_mask_fp_rate'],
        'healthy_val_fp_masks_per_image': healthy_val_fp['healthy_fp_masks_per_image'],
        'healthy_aware_labeled_val_mask_map50': val_score,
    }

    if RUN_TEST_EVALUATION:
        full_test = best_model.val(data=str(yaml_path), split='test', imgsz=TRAIN_IMGSZ, plots=True, verbose=False)
        labeled_test = best_model.val(data=str(labeled_eval_yaml), split='test', imgsz=TRAIN_IMGSZ, plots=False, verbose=False)
        test_count = count_prediction_errors(
            best_model,
            dataset_dir / 'test' / 'images',
            dataset_dir / 'test' / 'labels',
        )
        labeled_test_count = count_prediction_errors(
            best_model,
            labeled_eval_dir / 'test' / 'images',
            labeled_eval_dir / 'test' / 'labels',
        )
        healthy_test_fp = healthy_false_positive_summary(
            best_model,
            healthy_eval_dir / 'test' / 'images',
        )
        full_test_class_errors = image_level_class_error_summary(
            best_model,
            dataset_dir / 'test' / 'images',
            dataset_dir / 'test' / 'labels',
        )
        labeled_test_class_errors = image_level_class_error_summary(
            best_model,
            labeled_eval_dir / 'test' / 'images',
            labeled_eval_dir / 'test' / 'labels',
        )
        print_image_level_class_errors('Full test image-level class error summary', full_test_class_errors)
        print_image_level_class_errors('Labeled-only test image-level class error summary', labeled_test_class_errors)
        labeled_test_map50 = metric_value(labeled_test, 'seg.map50')
        row.update({
            'full_test_box_map50': metric_value(full_test, 'box.map50'),
            'full_test_mask_map50': metric_value(full_test, 'seg.map50'),
            'labeled_test_box_map50': metric_value(labeled_test, 'box.map50'),
            'labeled_test_mask_map50': labeled_test_map50,
            'labeled_test_mask_map50_95': metric_value(labeled_test, 'seg.map'),
            'test_gt_instances': test_count['gt_total'],
            'test_pred_boxes': test_count['pred_box_total'],
            'test_pred_masks': test_count['pred_mask_total'],
            'test_mask_count_mae': test_count['mask_count_mae'],
            'labeled_test_gt_instances': labeled_test_count['gt_total'],
            'labeled_test_pred_boxes': labeled_test_count['pred_box_total'],
            'labeled_test_pred_masks': labeled_test_count['pred_mask_total'],
            'labeled_test_mask_count_mae': labeled_test_count['mask_count_mae'],
            'labeled_test_mask_count_exact': labeled_test_count['mask_count_exact'],
            'labeled_test_disease_box_miss_rate': labeled_test_count['disease_box_miss_rate'],
            'labeled_test_disease_mask_miss_rate': labeled_test_count['disease_mask_miss_rate'],
            'healthy_test_images': healthy_test_fp['healthy_images'],
            'healthy_test_mask_fp_rate': healthy_test_fp['healthy_mask_fp_rate'],
            'healthy_test_box_fp_rate': healthy_test_fp['healthy_box_fp_rate'],
            'healthy_test_fp_masks_total': healthy_test_fp['healthy_fp_masks_total'],
            'healthy_test_fp_masks_per_image': healthy_test_fp['healthy_fp_masks_per_image'],
            'healthy_test_avg_fp_confidence': healthy_test_fp['healthy_avg_fp_confidence'],
            **flatten_image_level_class_errors('full_test', full_test_class_errors),
            **flatten_image_level_class_errors('labeled_test', labeled_test_class_errors),
            'healthy_aware_labeled_test_mask_map50': healthy_aware_score(labeled_test_map50, labeled_test_count, healthy_test_fp),
        })

    row.update(read_best_epoch_from_results(run_path))
    del yolo, best_model
    gc.collect()
    return row



# Hard gate: do not start any training until all paths, polygons and model shapes pass.
run_notebook_preflight(YOLO_MODELS, expected_attention={'DPCAGate': 3})

experiment_results = []
for model_name in YOLO_MODELS:
    YOLO_MODEL = model_name
    MODEL_STEM = EXPERIMENT_NAME
    RUN_BASE_NAME = EXPERIMENT_NAME
    for experiment in EXPERIMENTS:
        result = run_experiment(experiment)
        experiment_results.append(result)
        partial_df = pd.DataFrame(experiment_results)
        display(partial_df)
        partial_df.to_csv(REPORT_DIR / f'{EXPERIMENT_NAME}_results_partial.csv', index=False)

results_df = pd.DataFrame(experiment_results)
results_df = results_df.sort_values('healthy_aware_labeled_val_mask_map50', ascending=False).reset_index(drop=True)
summary_csv = REPORT_DIR / f'{EXPERIMENT_NAME}_results_summary.csv'
results_df.to_csv(summary_csv, index=False)
print(f'Saved summary: {summary_csv}')
display(results_df)

BEST_RUN = results_df.iloc[0].to_dict()
run_path = BEST_RUN['run_path']
best_model_path = BEST_RUN['best_pt']
print(f"Selected best run by validation healthy-aware score: {BEST_RUN['run_name']}")
print(f'Best checkpoint: {best_model_path}')


### Inspect Metrics, Overfitting, Counts, and Visual Samples
Use the summary table to inspect validation-selected strong-augmentation light-augmentation baseline behavior. Test metrics are reported but not used for model selection.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from IPython.display import display

if 'results_df' not in globals():
    results_df = pd.read_csv(REPORT_DIR / f'{EXPERIMENT_NAME}_results_summary.csv')
    BEST_RUN = results_df.iloc[0].to_dict()
    run_path = BEST_RUN['run_path']
    best_model_path = BEST_RUN['best_pt']

print('Strong-augmentation attention summary:')
display(results_df)

inspect_cols = [
    'experiment',
    'epochs_ran',
    'best_epoch_by_mask_map50',
    'best_val_mask_map50',
    'labeled_val_mask_map50',
    'labeled_val_mask_count_mae',
    'labeled_val_disease_box_miss_rate',
    'healthy_val_mask_fp_rate',
    'healthy_aware_labeled_val_mask_map50',
    'labeled_test_mask_map50',
    'labeled_test_mask_count_mae',
    'labeled_test_disease_box_miss_rate',
    'healthy_test_mask_fp_rate',
    'full_test_image_wrong_total',
    'full_test_image_wrong_rate',
    'full_test_healthy_wrong_images',
    'full_test_bg_wrong_images',
    'full_test_wssv_wrong_images',
    'labeled_test_image_wrong_total',
    'labeled_test_image_wrong_rate',
    'labeled_test_bg_wrong_images',
    'labeled_test_wssv_wrong_images',
    'healthy_aware_labeled_test_mask_map50',
]
print('Validation-selected inspection columns:')
display(results_df[[c for c in inspect_cols if c in results_df.columns]])

for _, row in results_df.iterrows():
    current_run_path = Path(row['run_path'])
    results_png = current_run_path / 'results.png'
    print(f"\nMetrics plot for {row['run_name']}: {results_png}")
    if results_png.exists():
        plt.figure(figsize=(12, 12))
        plt.imshow(mpimg.imread(results_png))
        plt.axis('off')
        plt.title(row['run_name'])
        plt.show()

print(f'Inspecting selected best run in: {run_path}')
val_batch = Path(run_path) / 'val_batch0_labels.jpg'
val_pred = Path(run_path) / 'val_batch0_pred.jpg'
fig, ax = plt.subplots(1, 2, figsize=(20, 10))
if val_batch.exists():
    ax[0].imshow(mpimg.imread(val_batch))
    ax[0].set_title('Validation Ground Truth Labels')
else:
    ax[0].set_title('Validation Labels not found')
if val_pred.exists():
    ax[1].imshow(mpimg.imread(val_pred))
    ax[1].set_title('Validation Predictions')
else:
    ax[1].set_title('Validation Predictions not found')
for a in ax:
    a.axis('off')
plt.show()


### Test Set Evaluation, Healthy False Positives, and Visual Inspection
This block evaluates the selected best checkpoint on:
- the full test set,
- the diseased/labeled-only test subset,
- the healthy/empty-label test subset for false positives.

It then visualizes augmented training masks, diseased test predictions, and healthy test predictions so you can inspect both missed disease masks and false alarms on healthy shrimp.

The notebook now also prints image-level class prediction errors for the full test set, the labeled-only test set, and healthy-negative images.


In [ ]:
from ultralytics import YOLO
import glob
import matplotlib.pyplot as plt
import os
import random
from pathlib import Path
import pandas as pd

model_inference = YOLO(best_model_path)
selected_exp = BEST_RUN['experiment']
selected_dataset_dir = EXPERIMENT_ROOT / selected_exp / 'dataset'
selected_yaml = selected_dataset_dir / 'data.yaml'
selected_labeled_yaml = EXPERIMENT_ROOT / selected_exp / 'dataset_labeled_only_eval' / 'data_labeled_only.yaml'
selected_healthy_dir = EXPERIMENT_ROOT / selected_exp / 'dataset_healthy_only_eval'
selected_healthy_yaml = selected_healthy_dir / 'data_healthy_only.yaml'

print('Full test-set evaluation:')
full_test_metrics = model_inference.val(data=str(selected_yaml), split='test', imgsz=640, plots=True, verbose=False)
print('Labeled-only diseased test-set evaluation:')
labeled_test_metrics = model_inference.val(data=str(selected_labeled_yaml), split='test', imgsz=640, plots=False, verbose=False)
healthy_test_fp = healthy_false_positive_summary(model_inference, selected_healthy_dir / 'test' / 'images')

print('Full test mask mAP50:', metric_value(full_test_metrics, 'seg.map50'))
print('Labeled-only diseased test mask mAP50:', metric_value(labeled_test_metrics, 'seg.map50'))
print('Healthy test mask false-positive rate:', healthy_test_fp['healthy_mask_fp_rate'])
print('Healthy test false-positive masks per image:', healthy_test_fp['healthy_fp_masks_per_image'])

print('\nImage-level class error summaries at prediction conf:', PREDICT_CONF_FOR_COUNT)
full_test_class_errors = image_level_class_error_summary(
    model_inference,
    selected_dataset_dir / 'test' / 'images',
    selected_dataset_dir / 'test' / 'labels',
)
labeled_test_class_errors = image_level_class_error_summary(
    model_inference,
    selected_labeled_yaml.parent / 'test' / 'images',
    selected_labeled_yaml.parent / 'test' / 'labels',
)
print_image_level_class_errors('Full test image-level class error summary', full_test_class_errors)
print_image_level_class_errors('Labeled-only test image-level class error summary', labeled_test_class_errors)

full_error_csv = REPORT_DIR / f'{EXPERIMENT_NAME}_full_test_image_class_errors.csv'
labeled_error_csv = REPORT_DIR / f'{EXPERIMENT_NAME}_labeled_test_image_class_errors.csv'
pd.DataFrame(full_test_class_errors['class_rows']).to_csv(full_error_csv, index=False)
pd.DataFrame(labeled_test_class_errors['class_rows']).to_csv(labeled_error_csv, index=False)
print('Saved full-test image class errors:', full_error_csv)
print('Saved labeled-test image class errors:', labeled_error_csv)


def draw_yolo_segmentation_labels(image_path, label_path):
    image = cv2.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]
    overlay = image.copy()
    colors = [(255, 70, 70), (70, 180, 255), (90, 220, 120), (240, 180, 60)]

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) < 7:
                continue
            cls_id = int(float(parts[0]))
            coords = np.array([float(v) for v in parts[1:]], dtype=np.float32).reshape(-1, 2)
            coords[:, 0] *= width
            coords[:, 1] *= height
            pts = coords.astype(np.int32)
            color = colors[cls_id % len(colors)]
            cv2.polylines(overlay, [pts], isClosed=True, color=color, thickness=2)
            cv2.fillPoly(overlay, [pts], color=color)
            x, y = pts[0]
            label = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
            cv2.putText(overlay, label, (int(x), int(y)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    return cv2.addWeighted(overlay, 0.35, image, 0.65, 0)


# Visualize augmented training ground-truth masks directly.
aug_images = sorted((selected_dataset_dir / 'train' / 'images').glob('aug_*'))
random.seed(42)
aug_images = random.sample(aug_images, k=min(6, len(aug_images)))

if aug_images:
    fig, axes = plt.subplots(len(aug_images), 1, figsize=(10, 5 * len(aug_images)))
    if len(aug_images) == 1:
        axes = [axes]
    for ax, image_path in zip(axes, aug_images):
        label_path = selected_dataset_dir / 'train' / 'labels' / f'{image_path.stem}.txt'
        ax.imshow(draw_yolo_segmentation_labels(image_path, label_path))
        ax.set_title(f'Augmented train GT mask: {image_path.name} | label exists={label_path.exists()}')
        ax.axis('off')
    plt.show()
else:
    print('No augmented training images found to visualize.')

# Visualize labeled test predictions.
test_images = []
for ext in IMAGE_EXTENSIONS:
    test_images.extend((selected_dataset_dir / 'test' / 'images').glob(f'*{ext}'))

def has_nonempty_label(image_path):
    label_path = selected_dataset_dir / 'test' / 'labels' / f'{image_path.stem}.txt'
    return label_path.exists() and bool(label_path.read_text().strip())

test_images = [p for p in sorted(test_images) if has_nonempty_label(p)]
test_images = test_images[:8]

results = model_inference.predict(source=[str(p) for p in test_images], conf=0.1, save=True, verbose=False)
for image_path, result in zip(test_images, results):
    plt.figure(figsize=(8, 8))
    plt.imshow(result.plot())
    plt.title(f'Labeled test prediction: {image_path.name} (conf > 0.1)')
    plt.axis('off')
    plt.show()

# Visualize healthy negative predictions to inspect false positives.
healthy_images = []
for ext in IMAGE_EXTENSIONS:
    healthy_images.extend((selected_healthy_dir / 'test' / 'images').glob(f'*{ext}'))
healthy_images = sorted(healthy_images)[:8]

if healthy_images:
    healthy_results = model_inference.predict(source=[str(p) for p in healthy_images], conf=0.1, save=False, verbose=False)
    for image_path, result in zip(healthy_images, healthy_results):
        plt.figure(figsize=(8, 8))
        plt.imshow(result.plot())
        box_count = len(result.boxes) if result.boxes is not None else 0
        mask_count = len(result.masks) if result.masks is not None else 0
        plt.title(f'Healthy test prediction: {image_path.name} | boxes={box_count}, masks={mask_count}')
        plt.axis('off')
        plt.show()
else:
    print('No healthy test images found to visualize.')


In [ ]:
print(f'Selected run path: {run_path}')
print(f'Selected checkpoint: {best_model_path}')
print(f'Report directory: {REPORT_DIR}')
print('Validation healthy-aware score = labeled_val_mask_map50 - COUNT_PENALTY_WEIGHT * labeled_val_mask_count_mae - DISEASE_MISS_PENALTY_WEIGHT * labeled_val_disease_box_miss_rate - HEALTHY_FP_PENALTY_WEIGHT * healthy_val_mask_fp_rate')
print('Strong augmentation = fliplr=0.5, flipud=0.3, hsv_h=0.05, hsv_s=0.5, hsv_v=0.4, degrees=10, translate=0.1, scale=0.5, erasing=0.15; mosaic/mixup/copy_paste/Albumentations disabled')
